In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Use relative project paths so the notebook works after cloning from GitHub.
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / "data" / "online_retail.xlsx"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
VISUALS_DIR = PROJECT_ROOT / "visuals"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place online_retail.xlsx in the data/ folder."
    )

df = pd.read_excel(DATA_PATH)

df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

# E-Commerce Customer Segmentation and Retention Analysis

## Project Overview

This project analyzes transaction-level data from an online retailer to identify customer segments, revenue drivers, and retention opportunities. The analysis focuses on cleaning raw transaction data, engineering customer-level RFM features, segmenting customers by purchasing behavior, and translating the results into marketing and retention recommendations.

## Business Questions

1. Which customers generate the most revenue?
2. Which customers purchase most frequently?
3. Which customers are high-value but at risk of churn?
4. Which products and countries contribute most to revenue?
5. How can customers be grouped into actionable marketing segments?

In [ ]:
# Make a copy before cleaning
retail = df.copy()

# Standardize column names
retail.columns = (
    retail.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

retail.head()

retail.info()


In [ ]:
# Count canceled invoices
retail["invoiceno"] = retail["invoiceno"].astype(str)

canceled_count = retail["invoiceno"].str.startswith("C").sum()
negative_quantity_count = (retail["quantity"] <= 0).sum()
nonpositive_price_count = (retail["unitprice"] <= 0).sum()
missing_customer_count = retail["customerid"].isna().sum()

print("Canceled invoices:", canceled_count)
print("Rows with quantity <= 0:", negative_quantity_count)
print("Rows with unit price <= 0:", nonpositive_price_count)
print("Rows missing customer ID:", missing_customer_count)

In [ ]:
retail["invoiceno"] = retail["invoiceno"].astype(str)
retail["invoicedate"] = pd.to_datetime(retail["invoicedate"], errors="coerce")
retail["quantity"] = pd.to_numeric(retail["quantity"], errors="coerce")
retail["unitprice"] = pd.to_numeric(retail["unitprice"], errors="coerce")

retail_clean = retail[
    (~retail["invoiceno"].str.startswith("C", na=False)) &
    (retail["quantity"] > 0) &
    (retail["unitprice"] > 0) &
    (retail["customerid"].notna()) &
    (retail["invoicedate"].notna())
].copy()

retail_clean["customerid"] = retail_clean["customerid"].astype(int)
retail_clean["revenue"] = retail_clean["quantity"] * retail_clean["unitprice"]

retail_clean.head()

In [ ]:
print("Original rows:", len(retail))
print("Cleaned rows:", len(retail_clean))
print("Rows removed:", len(retail) - len(retail_clean))

## Data Cleaning

The raw transaction data contains canceled invoices, negative quantities, non-positive prices, and transactions without customer identifiers. Since the goal is customer segmentation and retention analysis, I removed records that could not represent valid customer purchases.

Cleaning steps:
- Removed canceled invoices
- Removed rows with non-positive quantities
- Removed rows with non-positive unit prices
- Removed rows without customer IDs
- Created a `revenue` field as `quantity × unit price`

In [ ]:
cleaning_summary = pd.DataFrame({
    "metric": [
        "Original rows",
        "Cleaned rows",
        "Rows removed",
        "Unique customers",
        "Unique invoices",
        "Total revenue"
    ],
    "value": [
        len(retail),
        len(retail_clean),
        len(retail) - len(retail_clean),
        retail_clean["customerid"].nunique(),
        retail_clean["invoiceno"].nunique(),
        retail_clean["revenue"].sum()
    ]
})

cleaning_summary

In [ ]:
print("Cleaned data shape:", retail_clean.shape)
print("Unique customers:", retail_clean["customerid"].nunique())
print("Total revenue:", retail_clean["revenue"].sum())

## Customer-Level RFM Metrics

To segment customers, I created customer-level Recency, Frequency, and Monetary Value metrics.

- **Recency:** number of days since the customer's most recent purchase
- **Frequency:** number of unique invoices/purchases
- **Monetary value:** total revenue generated by the customer

These metrics are commonly used in customer analytics to identify loyal customers, high-value customers, and customers who may be at risk of churn.

In [ ]:
# Set reference date as one day after the most recent transaction
snapshot_date = retail_clean["invoicedate"].max() + pd.Timedelta(days=1)

customer_metrics = retail_clean.groupby("customerid").agg(
    recency=("invoicedate", lambda x: (snapshot_date - x.max()).days),
    frequency=("invoiceno", "nunique"),
    monetary_value=("revenue", "sum"),
    unique_products=("stockcode", "nunique"),
    first_purchase=("invoicedate", "min"),
    last_purchase=("invoicedate", "max")
).reset_index()

order_values = (
    retail_clean
    .groupby(["customerid", "invoiceno"])
    .agg(order_value=("revenue", "sum"))
    .reset_index()
)

customer_order_metrics = (
    order_values
    .groupby("customerid")
    .agg(
        avg_order_value=("order_value", "mean"),
        max_order_value=("order_value", "max")
    )
    .reset_index()
)

rfm = customer_metrics.merge(
    customer_order_metrics,
    on="customerid",
    how="left"
)

rfm["customer_lifetime_days"] = (
    rfm["last_purchase"] - rfm["first_purchase"]
).dt.days + 1

rfm.head()

In [ ]:
rfm.describe()

In [ ]:
rfm_summary = rfm[[
    "recency",
    "frequency",
    "monetary_value",
    "avg_order_value",
    "unique_products",
    "customer_lifetime_days"
]].describe().round(2)

rfm_summary

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(rfm["recency"], bins=40)
plt.title("Distribution of Customer Recency")
plt.xlabel("Days since last purchase")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "01_customer_recency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(np.log1p(rfm["frequency"]), bins=40)

plt.title("Distribution of Customer Purchase Frequency")
plt.xlabel("Log number of unique invoices")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "02_customer_frequency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(np.log1p(rfm["monetary_value"]), bins=40)

plt.title("Distribution of Customer Monetary Value")
plt.xlabel("Log total customer revenue")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "03_customer_monetary_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

Customer purchase frequency and monetary value are highly right-skewed. Most customers purchase only a small number of times and generate moderate revenue, while a smaller group of customers purchases frequently or generates much higher revenue.

To make these distributions easier to read, I plotted the log-transformed values using `log1p`, which applies `log(1 + x)`. This keeps customers with low values in the analysis while reducing the visual impact of extreme high-value customers.

In [ ]:
top_customers = rfm.sort_values("monetary_value", ascending=False).head(10)

top_customers[[
    "customerid",
    "recency",
    "frequency",
    "monetary_value",
    "avg_order_value",
    "unique_products"
]]

The RFM metrics show substantial variation in customer behavior. Some customers purchase frequently and generate high total revenue, while many customers purchase only a few times. This makes customer segmentation useful because different customer groups likely require different marketing and retention strategies.

## RFM Scoring and Customer Segmentation

After creating customer-level RFM metrics, I converted each metric into a score from 1 to 5. Customers with more recent purchases, higher purchase frequency, and higher monetary value receive higher scores.

These scores are used to create interpretable customer segments that can support marketing and retention decisions.

In [ ]:
rfm_scored = rfm.copy()

# Recency: lower recency is better, so labels are reversed
rfm_scored["r_score"] = pd.qcut(
    rfm_scored["recency"].rank(method="first"),
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# Frequency: higher frequency is better
rfm_scored["f_score"] = pd.qcut(
    rfm_scored["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary value: higher revenue is better
rfm_scored["m_score"] = pd.qcut(
    rfm_scored["monetary_value"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm_scored["rfm_score"] = (
    rfm_scored["r_score"] +
    rfm_scored["f_score"] +
    rfm_scored["m_score"]
)

rfm_scored.head()

In [ ]:
def assign_segment(row):
    if row["r_score"] >= 4 and row["f_score"] >= 4 and row["m_score"] >= 4:
        return "Champions"
    elif row["r_score"] >= 4 and row["f_score"] >= 3:
        return "Loyal Customers"
    elif row["r_score"] >= 4 and row["f_score"] <= 2 and row["m_score"] >= 4:
        return "Recent High-Value"
    elif row["r_score"] >= 4 and row["f_score"] <= 2:
        return "New/Potential Customers"
    elif row["r_score"] <= 2 and row["m_score"] >= 4:
        return "At-Risk High-Value"
    elif row["r_score"] <= 2 and row["f_score"] >= 3:
        return "At Risk"
    elif row["r_score"] <= 2 and row["f_score"] <= 2:
        return "Inactive/Low Engagement"
    else:
        return "Needs Attention"

rfm_scored["customer_segment"] = rfm_scored.apply(assign_segment, axis=1)


rfm_scored[
    [
        "customerid",
        "recency",
        "frequency",
        "monetary_value",
        "r_score",
        "f_score",
        "m_score",
        "rfm_score",
        "customer_segment"
    ]
].head()

In [ ]:
segment_summary = rfm_scored.groupby("customer_segment").agg(
    customers=("customerid", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary_value=("monetary_value", "mean"),
    avg_order_value=("avg_order_value", "mean"),
    total_revenue=("monetary_value", "sum")
).reset_index()

segment_summary["customer_share"] = segment_summary["customers"] / segment_summary["customers"].sum()
segment_summary["revenue_share"] = segment_summary["total_revenue"] / segment_summary["total_revenue"].sum()

segment_order = [
    "Champions",
    "Loyal Customers",
    "Recent High-Value",
    "New/Potential Customers",
    "Needs Attention",
    "At-Risk High-Value",
    "At Risk",
    "Inactive/Low Engagement"
]

segment_summary["customer_segment"] = pd.Categorical(
    segment_summary["customer_segment"],
    categories=segment_order,
    ordered=True
)

segment_summary = segment_summary.sort_values("customer_segment").reset_index(drop=True)

segment_summary.round(3)

In [ ]:
segment_counts = (
    rfm_scored["customer_segment"]
    .value_counts()
    .reindex(segment_order, fill_value=0)
)

plt.figure(figsize=(10, 5))
plt.bar(segment_counts.index, segment_counts.values)

plt.title("Customer Count by Segment")
plt.xlabel("Customer segment")
plt.ylabel("Number of customers")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "04_customer_count_by_segment.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
segment_revenue = (
    segment_summary
    .set_index("customer_segment")
    .reindex(segment_order)
    .fillna({"customers": 0, "total_revenue": 0})
    .reset_index()
)

plt.figure(figsize=(10, 5))
plt.bar(segment_revenue["customer_segment"], segment_revenue["total_revenue"])

plt.title("Total Revenue by Customer Segment")
plt.xlabel("Customer segment")
plt.ylabel("Total revenue")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(VISUALS_DIR / "05_revenue_by_customer_segment.png", dpi=150, bbox_inches="tight")
plt.show()

The RFM segmentation separates customers into groups with different business value and retention needs. Customers with high recency, frequency, and monetary value are classified as Champions, while customers with strong historical spending but low recent activity are classified as At-Risk High-Value. Customers who purchased recently and spent meaningfully, but have not yet purchased frequently, are classified as Recent High-Value.

The segment summary and charts use the same customer lifecycle order across both customer count and revenue views. This makes it easier to compare segment size against revenue contribution and identify important gaps. For example, some segments may contain many customers but contribute relatively little revenue, while smaller high-value segments may have an outsized impact on total sales.

This segmentation is useful because the same marketing strategy should not be applied to every customer. Champions and Loyal Customers may respond well to loyalty rewards, early access, or personalized offers. At-Risk High-Value customers are stronger candidates for win-back campaigns or reactivation messaging. Recent High-Value and New/Potential Customers may benefit from onboarding campaigns, product bundles, or second-purchase incentives.

## Product Revenue Analysis

In addition to customer segmentation, I analyzed product-level revenue to identify which products contribute the most to sales. This helps connect customer behavior to product performance and supports merchandising, cross-sell, and inventory recommendations.

In [ ]:
non_product_codes = ["POST", "M", "BANK CHARGES", "AMAZONFEE", "DOT", "CRUK"]

product_sales = retail_clean[
    ~retail_clean["stockcode"].astype(str).isin(non_product_codes)
].copy()

top_products = (
    product_sales
    .groupby(["stockcode", "description"])
    .agg(
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        unique_customers=("customerid", "nunique")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
    .head(10)
)

top_products

In [ ]:
def dollars_thousands(x, pos):
    return f"${x/1_000:.0f}K"

plt.figure(figsize=(10, 5))

plt.barh(
    top_products["description"].astype(str)[::-1],
    top_products["total_revenue"][::-1]
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Total revenue")
plt.ylabel("Product")
plt.gca().xaxis.set_major_formatter(FuncFormatter(dollars_thousands))
plt.tight_layout()
plt.savefig(VISUALS_DIR / "06_top_products_by_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

Non-product transaction codes such as postage and manual adjustments were excluded from the product ranking so that the analysis focuses on actual merchandise. 

The product ranking shows that revenue leadership can come from two different patterns: broad customer reach or unusually large purchases from a small number of customers. For example, `PAPER CRAFT, LITTLE BIRDIE` generates the highest product revenue but appears among only one unique customer, suggesting a bulk-purchase outlier. In contrast, `REGENCY CAKESTAND 3 TIER` and `WHITE HANGING HEART T-LIGHT HOLDER` combine high revenue with broad customer reach, making them stronger candidates for general promotions, merchandising focus, or cross-sell campaigns.

## Country Revenue Analysis

The dataset includes transactions from multiple countries. I summarized revenue, customer count, and order count by country to understand where sales are concentrated and whether the retailer depends heavily on specific geographic markets.

In [ ]:
country_summary = (
    retail_clean
    .groupby("country")
    .agg(
        total_revenue=("revenue", "sum"),
        unique_customers=("customerid", "nunique"),
        total_orders=("invoiceno", "nunique")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

country_summary.head(10)

In [ ]:
uk_revenue_share = (
    country_summary.loc[country_summary["country"] == "United Kingdom", "total_revenue"].iloc[0]
    / country_summary["total_revenue"].sum()
)

print("United Kingdom revenue share:", round(uk_revenue_share, 3))

The United Kingdom accounts for the majority of revenue in the cleaned transaction data, representing about 82% of total sales. This is expected because the retailer is UK-based, but it also shows that the business is highly concentrated in one domestic market.

Outside the UK, the Netherlands, EIRE, Germany, France, and Australia are the largest revenue contributors. Some international markets have high revenue despite relatively few unique customers, suggesting the presence of wholesale or high-value buyers rather than broad customer bases.

In [ ]:
def dollars_millions(x, pos):
    return f"${x/1_000_000:.1f}M"

top_countries = country_summary.head(10)

plt.figure(figsize=(10, 5))

plt.barh(
    top_countries["country"][::-1],
    top_countries["total_revenue"][::-1]
)

plt.title("Top 10 Countries by Revenue")
plt.xlabel("Total revenue")
plt.ylabel("Country")
plt.gca().xaxis.set_major_formatter(FuncFormatter(dollars_millions))
plt.tight_layout()
plt.savefig(VISUALS_DIR / "07_top_countries_by_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

## Revenue Concentration

To understand whether revenue depends heavily on a small share of customers, I calculated cumulative revenue contribution by customer rank. This helps identify whether the business relies disproportionately on its highest-value customers.

In [ ]:
rfm_revenue_ranked = rfm_scored.sort_values("monetary_value", ascending=False).copy()

rfm_revenue_ranked["revenue_rank"] = range(1, len(rfm_revenue_ranked) + 1)
rfm_revenue_ranked["cumulative_revenue"] = rfm_revenue_ranked["monetary_value"].cumsum()
rfm_revenue_ranked["cumulative_revenue_share"] = (
    rfm_revenue_ranked["cumulative_revenue"] / rfm_revenue_ranked["monetary_value"].sum()
)

top_10pct_cutoff = max(1, int(len(rfm_revenue_ranked) * 0.10))
top_20pct_cutoff = max(1, int(len(rfm_revenue_ranked) * 0.20))

top_10pct_revenue_share = (
    rfm_revenue_ranked
    .head(top_10pct_cutoff)["monetary_value"]
    .sum() / rfm_revenue_ranked["monetary_value"].sum()
)

top_20pct_revenue_share = (
    rfm_revenue_ranked
    .head(top_20pct_cutoff)["monetary_value"]
    .sum() / rfm_revenue_ranked["monetary_value"].sum()
)

print("Top 10% customer revenue share:", round(top_10pct_revenue_share, 3))
print("Top 20% customer revenue share:", round(top_20pct_revenue_share, 3))

The revenue concentration results show that customer value is highly uneven. The top 10% of customers generate 61.3% of total revenue, and the top 20% generate 74.6%. This means a relatively small share of customers accounts for most sales, making high-value customer retention a major business priority.

This finding supports the segmentation strategy: Champions and At-Risk High-Value customers should receive more focused attention than broad low-engagement segments because changes in their behavior can have a larger impact on total revenue.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    rfm_revenue_ranked["revenue_rank"],
    rfm_revenue_ranked["cumulative_revenue_share"]
)

plt.axvline(top_10pct_cutoff, linestyle="--", linewidth=1)
plt.axvline(top_20pct_cutoff, linestyle="--", linewidth=1)

label_offset = max(1, int(len(rfm_revenue_ranked) * 0.01))

plt.text(
    top_10pct_cutoff + label_offset,
    top_10pct_revenue_share - 0.04,
    f"Top 10%: {top_10pct_revenue_share:.1%}",
    ha="left",
    va="top"
)

plt.text(
    top_20pct_cutoff + label_offset,
    top_20pct_revenue_share + 0.02,
    f"Top 20%: {top_20pct_revenue_share:.1%}",
    ha="left",
    va="bottom"
)

plt.title("Cumulative Revenue Share by Customer Rank")
plt.xlabel("Customers ranked by monetary value")
plt.ylabel("Cumulative revenue share")
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.tight_layout()
plt.savefig(VISUALS_DIR / "08_cumulative_revenue_share.png", dpi=150, bbox_inches="tight")
plt.show()

The cumulative revenue curve rises steeply, showing that revenue is highly concentrated among the highest-value customers. The top 10% of customers generate 61.3% of total revenue, while the top 20% generate 74.6%. This supports a targeted retention strategy: protecting high-value customers is likely more valuable than treating all customers equally.

In [ ]:
# Export cleaned datasets and analysis tables for SQL, dashboards, and reporting.
TABLES_DIR.mkdir(parents=True, exist_ok=True)

retail_clean.to_csv(TABLES_DIR / "clean_transactions.csv", index=False)
cleaning_summary.to_csv(TABLES_DIR / "cleaning_summary.csv", index=False)
rfm_scored.to_csv(TABLES_DIR / "customer_rfm_segments.csv", index=False)
segment_summary.to_csv(TABLES_DIR / "segment_summary.csv", index=False)
top_products.to_csv(TABLES_DIR / "top_products.csv", index=False)
country_summary.to_csv(TABLES_DIR / "country_summary.csv", index=False)
rfm_revenue_ranked.to_csv(TABLES_DIR / "customer_revenue_ranked.csv", index=False)

print("Export complete.")
print(f"Tables saved to: {TABLES_DIR.resolve()}")


In [ ]:
top_segment = (
    segment_summary
    .sort_values("total_revenue", ascending=False)
    .iloc[0]["customer_segment"]
)

top_country = country_summary.iloc[0]["country"]

print("Project Summary")
print("----------------")
print(f"Cleaned transactions: {len(retail_clean):,}")
print(f"Unique customers: {retail_clean['customerid'].nunique():,}")
print(f"Total revenue: ${retail_clean['revenue'].sum():,.2f}")
print(f"Top revenue segment: {top_segment}")
print(f"Top revenue country: {top_country}")
print(f"Top 10% customer revenue share: {top_10pct_revenue_share:.1%}")
print(f"Top 20% customer revenue share: {top_20pct_revenue_share:.1%}")

## Key Findings

1. **Revenue is highly concentrated among top customers.** The top 10% of customers generate 61.3% of total revenue, and the top 20% generate 74.6%. This makes high-value customer retention a major business priority.

2. **Champions are the most valuable segment.** Customers classified as Champions represent a relatively small share of customers but generate the majority of total revenue.

3. **At-Risk High-Value customers are a major retention opportunity.** These customers have strong historical spending but lower recent activity, making them good candidates for win-back campaigns.

4. **Some high-revenue products are broad sellers, while others are outliers.** `PAPER CRAFT, LITTLE BIRDIE` generated the highest product revenue but was purchased by only one unique customer, suggesting a bulk-purchase outlier. Products such as `REGENCY CAKESTAND 3 TIER` and `WHITE HANGING HEART T-LIGHT HOLDER` combine high revenue with broad customer reach.

5. **Revenue is geographically concentrated in the United Kingdom.** The UK accounts for the majority of revenue, which is expected for a UK-based retailer but also shows dependence on one primary market.

## Business Recommendations

Based on the customer segmentation, product analysis, country analysis, and revenue concentration results, the retailer should prioritize marketing and retention efforts by customer value and recent activity.

1. **Protect Champions with loyalty-focused campaigns.**  
   Champions generate the largest share of revenue and should receive loyalty rewards, early access to new products, personalized recommendations, or VIP promotions. Retaining these customers should be a top priority because they have an outsized impact on total sales.

2. **Win back At-Risk High-Value customers.**  
   At-Risk High-Value customers have strong historical spending but lower recent activity. These customers should be targeted with reactivation emails, limited-time offers, personalized discounts, or product recommendations based on past purchases.

3. **Nurture Recent High-Value and New/Potential customers.**  
   Customers who purchased recently but have not yet developed high purchase frequency may become loyal customers with the right follow-up. Second-purchase incentives, onboarding campaigns, product bundles, and cross-sell recommendations could help move these customers toward higher-value segments.

4. **Avoid treating all inactive customers the same.**  
   Inactive/Low Engagement customers make up a meaningful share of the customer base but contribute less revenue. Broad campaigns to all inactive customers may be inefficient, so the retailer should prioritize inactive customers with higher historical monetary value first.

5. **Use product and country insights to guide targeting.**  
   Broad-reach products such as `REGENCY CAKESTAND 3 TIER` and `WHITE HANGING HEART T-LIGHT HOLDER` may be better candidates for general promotions, while outlier products such as `PAPER CRAFT, LITTLE BIRDIE` should be investigated separately as potential bulk-purchase behavior. Since revenue is heavily concentrated in the United Kingdom, international growth opportunities should be evaluated carefully by country-level customer value and order volume.

## Limitations

- The dataset covers transactions from 2010–2011, so findings are not intended to represent current e-commerce behavior.
- Customer segmentation is based only on transaction history and does not include demographics, acquisition channel, marketing exposure, website behavior, or customer satisfaction.
- RFM scoring uses relative rankings, so segment definitions depend on this specific dataset and chosen scoring rules.
- The analysis removes canceled invoices, invalid quantities, invalid prices, and transactions without customer IDs, so the segmentation focuses only on completed purchases linked to identifiable customers.
- Product analysis excludes non-product transaction codes such as postage and manual adjustments to focus on merchandise-level revenue.
- The analysis identifies associations and business patterns, but it does not test the causal effect of marketing actions on customer retention.

In [ ]:
# Create a simple business summary report for the report/ folder.
REPORT_DIR = PROJECT_ROOT / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

business_report = f"""# E-Commerce Customer Segmentation Business Summary

## Executive Summary

This analysis used online retail transaction data to segment customers using RFM analysis: recency, frequency, and monetary value. The main finding is that revenue is highly concentrated among a small share of customers. The top 10% of customers generated {top_10pct_revenue_share:.1%} of revenue, while the top 20% generated {top_20pct_revenue_share:.1%}.

## Key Metrics

- Cleaned transactions: {len(retail_clean):,}
- Unique customers: {retail_clean['customerid'].nunique():,}
- Total revenue: ${retail_clean['revenue'].sum():,.2f}
- Top revenue segment: {top_segment}
- Top country by revenue: {top_country}

## Recommended Actions

1. Prioritize retention for Champions and Loyal Customers.
2. Build win-back campaigns for At-Risk High-Value customers.
3. Use onboarding and second-purchase incentives for New/Potential Customers.
4. Avoid equal marketing spend across all customer groups.
5. Review product outliers before making merchandising decisions.
"""

(REPORT_DIR / "business_summary.md").write_text(business_report, encoding="utf-8")
print(f"Business summary saved to: {(REPORT_DIR / 'business_summary.md').resolve()}")


## Conclusion

This project demonstrates how transaction-level retail data can be transformed into customer-level business insights. By cleaning raw transaction records, engineering RFM metrics, segmenting customers, and analyzing product, country, and revenue concentration patterns, the analysis identifies which customers are most valuable, which customers may be at risk, and where marketing efforts can be targeted more efficiently.

The main business takeaway is that customer value is highly uneven. The top 10% of customers generate 61.3% of revenue, and the top 20% generate 74.6%. This makes high-value customer retention especially important. A targeted strategy focused on Champions, At-Risk High-Value customers, and promising recent buyers is likely more efficient than treating all customers the same.